# Lesson 5 — The GWTC Catalog: Exploring Real Events

**Gravitational-Wave Data Analysis with Python**  
LVK Python Course — Module 5

> *"The catalog is not just a list of events — it is a census of the compact-object universe."*

---

## Table of Contents

1. [Introduction & Motivation](#1.-Introduction-&-Motivation)  
2. [The GWTC Catalog Series](#2.-The-GWTC-Catalog-Series)  
   - 2.1 GWTC-1 (O1 + O2)  
   - 2.2 GWTC-2 and GWTC-2.1 (O3a)  
   - 2.3 GWTC-3 (O3b)  
   - 2.4 Towards GWTC-4 (O4)  
   - 2.5 Search Pipelines and False-Alarm Rate  
3. [Programmatic Catalog Access](#3.-Programmatic-Catalog-Access)  
   - 3.1 The `gwosc` Python Package  
   - 3.2 `pycbc.catalog`  
   - 3.3 Building a Unified Event Table  
4. [Key Physical Parameters](#4.-Key-Physical-Parameters)  
   - 4.1 Component Masses $m_1$, $m_2$  
   - 4.2 Chirp Mass $\mathcal{M}$ and Mass Ratio $q$  
   - 4.3 Effective Inspiral Spin $\chi_{\rm eff}$  
   - 4.4 Luminosity Distance $d_L$ and Redshift $z$  
   - 4.5 Network Signal-to-Noise Ratio  
5. [Population-Level Distributions](#5.-Population-Level-Distributions)  
   - 5.1 Mass Spectrum  
   - 5.2 Distance and Redshift Distribution  
   - 5.3 SNR and Detection Sensitivity  
   - 5.4 Spin Distribution  
6. [Reproducing "Masses in the Stellar Graveyard"](#6.-Reproducing-"Masses-in-the-Stellar-Graveyard")  
7. [Accessing Posterior Samples (HDF5)](#7.-Accessing-Posterior-Samples-(HDF5))  
   - 7.1 HDF5 File Structure  
   - 7.2 Loading and Inspecting Posterior Data  
   - 7.3 Corner Plots  
   - 7.4 Computing Derived Quantities  
8. [Advanced Topics](#8.-Advanced-Topics)  
   - 8.1 The Mass Gap(s)  
   - 8.2 Population Inference  
   - 8.3 Gravitational-Wave Transient Source Classes  
9. [Student Exercises](#9.-Student-Exercises)  
10. [References](#10.-References)  


---
## 1. Introduction & Motivation

The **Gravitational-Wave Transient Catalog (GWTC)** is a living document: each new observing run adds dozens of confident detections and extends our window on the compact-object universe. By the end of GWTC-3 (2023), the LVK collaboration had reported **~90 candidate events**, spanning binary black hole (BBH), binary neutron star (BNS), and neutron star–black hole (NSBH) mergers.

Analysing the catalog as a whole — rather than event by event — allows us to address questions that individual detections cannot answer:

| Science question | Observable |
|---|---|
| What is the black hole mass function? | $m_1$, $m_2$ distribution |
| Is there a mass gap between NS and BH? | $m_1$, $m_2$ in the 2–5 $M_\odot$ range |
| How fast do compact objects spin? | $\chi_{\rm eff}$, $\chi_p$ distribution |
| How does the merger rate evolve with redshift? | $d_L$ or $z$ distribution |
| What is the binary formation channel? | Correlated mass–spin–distance trends |
| What is the Hubble constant? | $d_L$ vs. redshift (standard sirens) |

In this lesson we will:
1. Understand the catalog structure and the search pipelines that produce it.
2. Query the catalog programmatically using `gwosc` and `pycbc.catalog`.
3. Extract and visualise key event parameters.
4. Reproduce the iconic *Masses in the Stellar Graveyard* plot.
5. Access full posterior distributions from HDF5 files released on GWOSC.

### Prerequisites
- Lessons 1–4 (GW physics, data access, time-series, spectral analysis)  
- Basic familiarity with `numpy`, `matplotlib`, and `pandas`  
- Optional: `h5py` or `pesummary` for posterior samples  


In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import warnings
warnings.filterwarnings('ignore')

# pandas for tabular data
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("pandas not installed — install with: pip install pandas")

# gwosc — open-science catalog API
try:
    import gwosc
    from gwosc.datasets import find_datasets, event_gps
    from gwosc.catalog import events as gwosc_events
    HAS_GWOSC = True
    print(f"gwosc version: {gwosc.__version__}")
except ImportError:
    HAS_GWOSC = False
    print("gwosc not installed — install with: pip install gwosc")

# pycbc catalog
try:
    from pycbc import catalog as pycbc_catalog
    HAS_PYCBC_CAT = True
    print("pycbc.catalog available")
except ImportError:
    HAS_PYCBC_CAT = False
    print("pycbc not installed — install with: pip install pycbc")

# h5py for posterior HDF5 files
try:
    import h5py
    HAS_H5PY = True
except ImportError:
    HAS_H5PY = False
    print("h5py not installed — install with: pip install h5py")

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
rng = np.random.default_rng(seed=42)
print("NumPy:", np.__version__)
print("Environment ready.")


---
## 2. The GWTC Catalog Series

The GWTC catalogs are the primary observational data products of the LVK collaboration. They compile:
- **Detection candidates** passing a false-alarm-rate (FAR) threshold
- **Median posterior estimates** of source parameters (masses, spins, distance)
- **Full posterior samples** packaged in HDF5 format
- **Strain data** for confirmed events released on GWOSC

### 2.1 GWTC-1 — First and Second Observing Runs (O1 + O2)

| Run | Dates | Detectors | Events |
|---|---|---|---|
| O1 | Sep 2015 – Jan 2016 | H1, L1 | 3 (GW150914, GW151012, GW151226) |
| O2 | Nov 2016 – Aug 2017 | H1, L1, V1 (from Aug 2017) | 8 incl. GW170817 |

**Landmark events:**
- **GW150914** — First ever detected GW; BBH merger at $d_L \approx 410$ Mpc; $m_1 \approx 36\,M_\odot$, $m_2 \approx 29\,M_\odot$.  
- **GW170817** — First BNS merger; associated with kilonova AT2017gfo, gamma-ray burst GRB 170817A, and a Hubble constant measurement.

### 2.2 GWTC-2 and GWTC-2.1 — Third Observing Run, Part A (O3a)

O3a ran from April–September 2019. GWTC-2 added **39 new candidates** for a total of 47, with improved sensitivity in Virgo. GWTC-2.1 (2021) re-analysed O1+O2+O3a with updated pipelines, reporting **55 candidates**.

Notable events:
- **GW190412** — First confident asymmetric-mass BBH with $q \approx 0.28$; shows higher-order modes.
- **GW190425** — Likely BNS with total mass $\approx 3.4\,M_\odot$ (heavier than any known NS binary).
- **GW190521** — Heaviest BBH; final mass $\approx 150\,M_\odot$ (intermediate mass BH!); both component masses in the pair-instability mass gap.
- **GW190814** — NSBH candidate; secondary mass $\approx 2.6\,M_\odot$ in the lower mass gap.

### 2.3 GWTC-3 — Third Observing Run, Part B (O3b)

O3b ran from November 2019 – March 2020. GWTC-3 reported **90 candidate events** in total from O1–O3, making it the most comprehensive catalog to date.

Key statistics from GWTC-3:

| Source class | Number of events |
|---|---|
| BBH (confident) | ~83 |
| BNS | 2 (GW170817, GW190425) |
| NSBH | 2 (GW200105, GW200115) |
| Ambiguous / marginal | ~35 |

### 2.4 Towards GWTC-4 — Fourth Observing Run (O4)

O4 began in May 2023 and is ongoing (O4a: May–Jan 2024; O4b: April 2024 onward). With improved detector sensitivity, O4 is expected to detect **several events per week**, increasing the catalog to hundreds of events. GWOSC provides O4 data and public alerts via GraceDB.

### 2.5 Search Pipelines and False-Alarm Rate

Events in the catalog are identified by matched-filter search pipelines:

| Pipeline | Type | Key reference |
|---|---|---|
| **PyCBC** | Modelled, matched filter | Nitz et al. 2020 |
| **GstLAL** | Modelled, matched filter | Messick et al. 2017 |
| **MBTA** | Modelled, matched filter | Aubin et al. 2021 |
| **cWB** | Unmodelled, coherent WaveBurst | Klimenko et al. 2016 |
| **oLIB** | Unmodelled, Q-transform based | Lynch et al. 2017 |

The **false-alarm rate (FAR)** quantifies how often a noise fluctuation mimics the observed event. GWTC-3 uses $\text{FAR} < 2\,\text{yr}^{-1}$ as the threshold for catalog inclusion, corresponding roughly to $p_{\rm astro} > 0.5$.  

The **pastro** (probability of astrophysical origin) is computed per event as:

$$
p_{\rm astro} = \frac{R_{\rm signal}\, \mathcal{L}(d|\text{signal})}{R_{\rm signal}\, \mathcal{L}(d|\text{signal}) + R_{\rm noise}\, \mathcal{L}(d|\text{noise})}
$$

where $R_{\rm signal}$ and $R_{\rm noise}$ are the expected rates of astrophysical signals and noise events respectively.


---
## 3. Programmatic Catalog Access

### 3.1 The `gwosc` Python Package

The `gwosc` package is the official Python interface to the **Gravitational Wave Open Science Center** API. It allows programmatic access to datasets, event GPS times, strain data URLs, and catalog information.

Key functions:

| Function | Description |
|---|---|
| `gwosc.datasets.find_datasets()` | List available datasets/catalogs |
| `gwosc.catalog.events()` | List events in a given catalog |
| `gwosc.datasets.event_gps(name)` | GPS time of a named event |
| `gwosc.locate.get_urls(det, start, end)` | URLs for strain data |
| `gwosc.event_detectors(name)` | Detectors that observed an event |

### 3.2 `pycbc.catalog`

`pycbc.catalog` provides a convenient object-oriented interface to GWTC data. The `Catalog` class loads event metadata and the `MergerSeries` object provides direct access to strain data.

### 3.3 Building a Unified Event Table

A common workflow is to build a pandas DataFrame collecting key parameters for all events, enabling population-level analysis.


In [ ]:
# ── 3.1 Explore available datasets with gwosc ──────────────────────────────
if HAS_GWOSC:
    # List all available catalogs/datasets
    all_datasets = find_datasets(type='catalog')
    print("Available catalogs on GWOSC:")
    for ds in sorted(all_datasets):
        print(f"  {ds}")


In [ ]:
# ── 3.1 List events in GWTC-3 ─────────────────────────────────────────────
if HAS_GWOSC:
    # Get events from GWTC-3
    gwtc3_events = gwosc_events('GWTC-3')
    print(f"\nGWTC-3 contains {len(gwtc3_events)} events.")
    print("First 10 events:", list(gwtc3_events)[:10])


In [ ]:
# ── 3.2 pycbc.catalog — build a Catalog object ────────────────────────────
if HAS_PYCBC_CAT:
    cat = pycbc_catalog.Catalog(source='gwtc-2')
    print(f"pycbc GWTC-2 catalog: {len(cat)} events")

    # Show available parameters for GW150914
    m = cat['GW150914']
    print("\nGW150914 median parameters:")
    print(f"  m1        = {m.median1d('mass1'):>6.2f} Msun")
    print(f"  m2        = {m.median1d('mass2'):>6.2f} Msun")
    print(f"  Mc        = {m.median1d('mchirp'):>6.2f} Msun")
    print(f"  chi_eff   = {m.median1d('chi_eff'):>6.3f}")
    print(f"  dL        = {m.median1d('distance'):>6.1f} Mpc")


In [ ]:
# ── 3.3 Build a unified event table ───────────────────────────────────────
#
# We assemble a DataFrame of median parameter estimates for the
# confident events in GWTC-3 that are publicly accessible.
# If pycbc.catalog is available we use it; otherwise we fall back
# to a curated subset hard-coded here for demonstration.

# Curated table: ~50 representative GWTC-3 events
# Columns: name, m1 [Msun], m2 [Msun], Mc [Msun], q, chi_eff, dL [Mpc], network_snr, pastro
_catalog_data = [
    # name               m1      m2     Mc      q       chi_eff   dL    snr   pastro
    ('GW150914',        35.6,   30.6,  28.3,  0.86,   -0.01,   440,  24.4,  1.00),
    ('GW151012',        23.3,   13.6,  15.2,  0.58,    0.05,  1060,   9.7,  0.97),
    ('GW151226',        13.7,    7.7,   8.9,  0.56,    0.18,   440,  13.1,  1.00),
    ('GW170104',        31.0,   20.1,  21.5,  0.65,   -0.04,   880,  13.0,  1.00),
    ('GW170608',        11.0,    7.6,   7.9,  0.69,    0.03,   320,  14.9,  1.00),
    ('GW170729',        50.2,   34.0,  35.7,  0.68,    0.37,  2750,  10.8,  0.98),
    ('GW170809',        35.2,   23.8,  25.0,  0.68,    0.07,  1000,  12.4,  1.00),
    ('GW170814',        30.7,   25.3,  24.2,  0.82,    0.07,   600,  15.9,  1.00),
    ('GW170817',         1.46,   1.27,  1.19,  0.87,   0.00,    40,  32.4,  1.00),
    ('GW170818',        35.5,   26.8,  26.5,  0.76,   -0.09,  1020,  11.3,  1.00),
    ('GW170823',        39.6,   29.4,  29.3,  0.74,    0.09,  1850,  11.5,  1.00),
    ('GW190408_181802', 24.6,   18.4,  18.3,  0.75,   -0.03,  1540,  14.8,  1.00),
    ('GW190412',        30.1,    8.3,  13.3,  0.28,    0.05,   740,  19.8,  1.00),
    ('GW190425',         2.0,    1.4,   1.45,  0.70,   0.05,   156,  12.9,  0.95),
    ('GW190503_185404', 42.0,   28.6,  30.3,  0.68,    0.07,  1930,  11.1,  0.97),
    ('GW190512_180714', 23.3,   12.6,  14.9,  0.54,    0.03,  1493,  12.4,  0.99),
    ('GW190513_205428', 35.7,   18.0,  21.6,  0.50,    0.11,  2200,  12.6,  0.97),
    ('GW190514_065416', 41.4,   28.7,  31.2,  0.69,   -0.06,  2500,   8.8,  0.73),
    ('GW190517_055101', 37.9,   25.3,  26.8,  0.67,    0.52,  2900,  11.1,  0.96),
    ('GW190519_153544', 66.0,   40.5,  44.6,  0.61,    0.31,  3000,  13.3,  0.98),
    ('GW190521',        95.3,   69.0,  72.3,  0.72,    0.07,  5300,  14.7,  0.97),
    ('GW190521_074359', 42.1,   32.8,  32.7,  0.78,    0.08,  1200,  24.7,  1.00),
    ('GW190527_092055', 36.5,   20.3,  24.1,  0.56,    0.07,  2700,   8.7,  0.74),
    ('GW190602_175927', 68.0,   51.3,  52.0,  0.75,    0.06,  3500,  12.9,  0.97),
    ('GW190620_030421', 60.1,   38.1,  41.4,  0.63,    0.33,  3100,  10.9,  0.93),
    ('GW190630_185205', 38.0,   23.9,  27.2,  0.63,    0.10,  1950,  15.8,  1.00),
    ('GW190701_203306', 54.8,   43.7,  44.0,  0.80,    0.13,  3000,  11.7,  0.99),
    ('GW190706_222641', 67.0,   38.2,  44.9,  0.57,    0.14,  4400,  13.0,  0.97),
    ('GW190707_093326', 12.1,    7.6,   8.5,  0.63,    0.10,   790,  13.0,  1.00),
    ('GW190708_232457', 18.2,   13.2,  13.8,  0.73,    0.18,   760,  13.4,  1.00),
    ('GW190720_000836', 14.0,    7.9,   9.5,  0.57,    0.18,   869,  11.8,  0.98),
    ('GW190727_060333', 38.6,   26.6,  28.7,  0.69,    0.07,  2900,  12.0,  0.99),
    ('GW190728_064510', 12.3,    8.1,   8.8,  0.66,    0.12,   880,  13.7,  1.00),
    ('GW190814',        23.2,    2.6,   6.1,  0.11,   -0.05,   241,  25.0,  0.99),
    ('GW190828_063405', 32.1,   26.1,  25.8,  0.81,    0.04,  1590,  17.2,  1.00),
    ('GW190828_065509', 24.1,   11.1,  14.7,  0.46,    0.06,  1800,  10.5,  0.95),
    ('GW190910_112807', 44.1,   32.9,  34.2,  0.75,    0.04,  2720,  14.7,  0.99),
    ('GW190915_235702', 35.3,   24.5,  25.8,  0.69,   -0.01,  1700,  13.8,  1.00),
    ('GW190924_021846',  8.9,    5.0,   5.9,  0.56,    0.08,   550,  13.2,  1.00),
    ('GW190929_012149', 68.3,   38.1,  45.4,  0.56,    0.20,  5600,  10.3,  0.87),
    ('GW190930_133541', 12.3,    7.3,   8.5,  0.59,    0.41,   808,  10.2,  0.74),
    ('GW191105_143521', 11.1,    8.3,   8.5,  0.75,    0.11,   980,  12.7,  0.98),
    ('GW191109_010717', 65.0,   47.1,  48.3,  0.72,   -0.29,  1600,  15.5,  0.99),
    ('GW191204_171526', 12.0,    7.7,   8.5,  0.64,    0.12,   694,  17.4,  1.00),
    ('GW191215_223052', 24.8,   15.0,  17.2,  0.60,    0.07,  1580,  10.7,  0.90),
    ('GW191216_213338', 12.1,    7.7,   8.6,  0.64,    0.22,   341,  18.6,  1.00),
    ('GW191222_033537', 47.8,   35.1,  37.0,  0.73,    0.09,  4000,  12.8,  0.99),
    ('GW191230_180458', 49.5,   36.4,  38.4,  0.74,    0.14,  4700,  10.5,  0.73),
    ('GW200105_162426',  8.9,    1.9,   3.4,  0.21,   -0.01,   280,  13.9,  0.97),
    ('GW200115_042309',  5.9,    1.44,  2.4,  0.24,   -0.15,   340,  11.6,  0.99),
    ('GW200129_065458', 36.0,   29.0,  28.2,  0.81,    0.06,  920,  26.8,  1.00),
    ('GW200202_154313', 10.1,    7.3,   7.9,  0.72,    0.09,   410,  12.9,  1.00),
    ('GW200208_130117', 46.4,   30.0,  34.1,  0.65,    0.19,  3400,  10.8,  0.97),
    ('GW200209_085452', 46.8,   31.9,  34.8,  0.68,    0.18,  4500,  10.1,  0.72),
    ('GW200219_094415', 39.0,   25.4,  28.4,  0.65,    0.11,  3750,  10.7,  0.91),
    ('GW200224_222234', 40.0,   32.4,  32.2,  0.81,    0.11,  1700,  20.0,  1.00),
    ('GW200225_060421', 19.3,   13.9,  14.8,  0.72,   -0.06,  1100,  12.9,  1.00),
    ('GW200302_015811', 28.1,   15.2,  18.8,  0.54,    0.10,  2580,   9.9,  0.83),
    ('GW200311_115853', 37.6,   29.7,  29.7,  0.79,    0.04,  1100,  17.8,  1.00),
    ('GW200316_215756', 13.1,    7.8,   9.1,  0.59,    0.07,   1130, 10.1,  0.79),
]

columns = ['name', 'm1', 'm2', 'Mc', 'q', 'chi_eff', 'dL', 'snr', 'pastro']

if HAS_PANDAS:
    df = pd.DataFrame(_catalog_data, columns=columns)
    # Derived quantities
    df['Mtotal'] = df['m1'] + df['m2']
    df['Mfinal'] = 0.952 * df['Mtotal']   # rough estimate
    df['z_approx'] = df['dL'] / 4425.0    # rough H0=70, Om=0.3 approx
    print(f"Event table built: {len(df)} rows")
    display(df.head(10))
else:
    # numpy fallback
    arr = np.array([(r[1], r[2], r[3], r[4], r[5], r[6], r[7], r[8])
                    for r in _catalog_data],
                   dtype=float)
    names_arr = [r[0] for r in _catalog_data]
    print(f"Event table built: {len(arr)} events (pandas not available)")


---
## 4. Key Physical Parameters

### 4.1 Component Masses $m_1$, $m_2$

By convention, $m_1 \geq m_2$ (primary and secondary mass). In geometrised units, masses are often expressed in solar masses $M_\odot = 1.989 \times 10^{30}$ kg.

Because GWs travel cosmological distances, what we measure is the **redshifted (detector-frame) mass**:

$$
m_i^{\rm det} = (1 + z)\, m_i^{\rm source}
$$

Converting to source-frame masses requires knowledge of the luminosity distance $d_L$ and a cosmological model.

### 4.2 Chirp Mass $\mathcal{M}$ and Mass Ratio $q$

The **chirp mass** is the single best-measured mass parameter from the gravitational waveform. It governs the leading-order frequency evolution of the inspiral:

$$
\mathcal{M} = \frac{(m_1 m_2)^{3/5}}{(m_1 + m_2)^{1/5}}
$$

The **mass ratio** is defined as:

$$
q = \frac{m_2}{m_1} \leq 1
$$

Given $\mathcal{M}$ and $q$, the component masses are:

$$
m_1 = \mathcal{M}\, \frac{(1+q)^{1/5}}{q^{3/5}}, \qquad m_2 = q\, m_1
$$

The chirp mass can be measured to sub-percent precision, while $q$ and individual masses typically have 10–30% uncertainties.

### 4.3 Effective Inspiral Spin $\chi_{\rm eff}$

The **effective spin parameter** is the mass-weighted projection of both component spins onto the orbital angular momentum:

$$
\chi_{\rm eff} = \frac{m_1 \chi_1^z + m_2 \chi_2^z}{m_1 + m_2}
$$

where $\chi_i^z = a_i \cos\theta_i$ (dimensionless spin magnitude $a_i \in [0,1]$, tilt angle $\theta_i$).  
Range: $\chi_{\rm eff} \in [-1, +1]$.  
- $\chi_{\rm eff} > 0$: net spin aligned with orbit (more common in isolated binary evolution)  
- $\chi_{\rm eff} \approx 0$: spin-orbit misalignment or small spins (expected in dynamical capture)

The **precessing spin** $\chi_p$ captures in-plane spin components responsible for orbital precession; it is generally less well-measured.

### 4.4 Luminosity Distance $d_L$ and Redshift $z$

GWs are **standard sirens**: the amplitude directly encodes $d_L$ without the distance ladder. The relationship between $d_L$ and $z$ depends on cosmology. For a flat $\Lambda$CDM universe:

$$
d_L(z) = \frac{c(1+z)}{H_0} \int_0^z \frac{dz'}{\sqrt{\Omega_m (1+z')^3 + \Omega_\Lambda}}
$$

With $H_0 = 70$ km/s/Mpc, $\Omega_m = 0.3$:  
- $d_L = 100$ Mpc $\Rightarrow$ $z \approx 0.022$  
- $d_L = 1$ Gpc $\Rightarrow$ $z \approx 0.20$  
- $d_L = 5$ Gpc $\Rightarrow$ $z \approx 0.78$  

### 4.5 Network Signal-to-Noise Ratio

The **network SNR** is the quadrature sum of individual detector SNRs:

$$
\rho_{\rm net} = \sqrt{\sum_i \rho_i^2}
$$

For Gaussian noise, the detection threshold is typically $\rho_{\rm net} \gtrsim 8$–12. The SNR also sets the statistical precision of parameter estimation: uncertainties scale roughly as $\sigma \propto 1/\rho$.


In [ ]:
# ── 4. Parameter distributions: individual events ─────────────────────────
if HAS_PANDAS:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.ravel()

    # m1 vs m2 scatter
    sc = axes[0].scatter(df['m2'], df['m1'],
                         c=df['dL'], cmap='viridis',
                         s=60, alpha=0.8, edgecolors='k', lw=0.3)
    axes[0].set(xlabel=r'$m_2\,[M_\odot]$', ylabel=r'$m_1\,[M_\odot]$',
                title='Component masses')
    axes[0].plot([0, 100], [0, 100], 'k--', alpha=0.3, label='$m_1=m_2$')
    axes[0].legend(fontsize=9)
    plt.colorbar(sc, ax=axes[0], label=r'$d_L$ [Mpc]')

    # Chirp mass histogram
    axes[1].hist(df['Mc'], bins=20, color='C0', edgecolor='k', alpha=0.75)
    axes[1].set(xlabel=r'Chirp mass $\mathcal{M}\,[M_\odot]$',
                ylabel='Number of events',
                title='Chirp mass distribution')

    # Mass ratio histogram
    axes[2].hist(df['q'], bins=15, color='C1', edgecolor='k', alpha=0.75)
    axes[2].set(xlabel=r'Mass ratio $q = m_2/m_1$',
                ylabel='Number of events',
                title='Mass ratio distribution')

    # chi_eff histogram
    axes[3].hist(df['chi_eff'], bins=20, color='C2', edgecolor='k', alpha=0.75)
    axes[3].axvline(0, color='k', ls='--', alpha=0.5)
    axes[3].set(xlabel=r'Effective spin $\chi_{\rm eff}$',
                ylabel='Number of events',
                title='Effective spin distribution')

    # Luminosity distance histogram
    axes[4].hist(df['dL'], bins=20, color='C3', edgecolor='k', alpha=0.75)
    axes[4].set(xlabel=r'Luminosity distance $d_L$ [Mpc]',
                ylabel='Number of events',
                title='Distance distribution')

    # Network SNR histogram
    axes[5].hist(df['snr'], bins=15, color='C4', edgecolor='k', alpha=0.75)
    axes[5].axvline(8, color='r', ls='--', alpha=0.6, label='SNR = 8')
    axes[5].set(xlabel=r'Network SNR $\rho$',
                ylabel='Number of events',
                title='SNR distribution')
    axes[5].legend()

    plt.suptitle('GWTC-3 — Key Parameter Distributions', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 4.2  Chirp mass formula: verification ─────────────────────────────────
def chirp_mass(m1, m2):
    """Compute chirp mass from component masses."""
    return (m1 * m2)**0.6 / (m1 + m2)**0.2

def component_masses(Mc, q):
    """Invert to component masses from (Mc, q)."""
    m1 = Mc * (1 + q)**0.2 / q**0.6
    m2 = q * m1
    return m1, m2

# Verify for GW150914
m1_v, m2_v = 35.6, 30.6
Mc_v = chirp_mass(m1_v, m2_v)
q_v  = m2_v / m1_v
print(f"GW150914: m1={m1_v}, m2={m2_v}")
print(f"  -> Mc = {Mc_v:.2f} Msun  (catalog: 28.3)")
print(f"  -> q  = {q_v:.3f}        (catalog: 0.86)")

# Demonstrate mass-parameter degeneracy
Mc_fixed = 28.3
q_range  = np.linspace(0.1, 1.0, 200)
m1_arr, m2_arr = component_masses(Mc_fixed, q_range)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(q_range, m1_arr, label=r'$m_1$', lw=2)
ax.plot(q_range, m2_arr, label=r'$m_2$', lw=2)
ax.axvline(q_v, color='grey', ls=':', alpha=0.7, label=f'GW150914 q={q_v:.2f}')
ax.set(xlabel='Mass ratio $q$',
       ylabel=r'Mass $[M_\odot]$',
       title=fr'Component masses for fixed $\mathcal{{M}}={Mc_fixed}\,M_\odot$')
ax.legend()
plt.tight_layout()
plt.show()
print("Note: for fixed Mc, many (m1, m2) pairs are possible — q is less well measured.")


In [ ]:
# ── 4.4 Luminosity distance — redshift conversion ─────────────────────────
from scipy.integrate import quad

H0   = 70.0          # km/s/Mpc
c_km = 2.998e5       # km/s
Om   = 0.3
Ol   = 0.7

def luminosity_distance(z, H0=70.0, Om=0.3):
    """Flat LCDM luminosity distance [Mpc] for scalar redshift z."""
    Ol = 1.0 - Om
    integrand = lambda zp: 1.0 / np.sqrt(Om * (1 + zp)**3 + Ol)
    comoving, _ = quad(integrand, 0, z)
    return c_km * (1 + z) / H0 * comoving

# Vectorised wrapper
def dL_vec(z_arr):
    return np.array([luminosity_distance(zi) for zi in z_arr])

z_arr  = np.linspace(0.001, 2.0, 500)
dL_arr = dL_vec(z_arr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(z_arr, dL_arr / 1e3)
axes[0].set(xlabel='Redshift $z$',
            ylabel=r'$d_L$ [Gpc]',
            title=r'Luminosity distance vs. redshift ($H_0=70$, $\Omega_m=0.3$)')

if HAS_PANDAS:
    axes[1].scatter(df['z_approx'], df['dL'], alpha=0.7, s=50, edgecolors='k', lw=0.3)
    axes[1].set(xlabel='Approx. redshift $z$',
                ylabel=r'$d_L$ [Mpc]',
                title='GWTC-3 events: $d_L$ vs $z$')

plt.tight_layout()
plt.show()

# Print some benchmarks
for z_test in [0.1, 0.5, 1.0, 2.0]:
    print(f"  z = {z_test:.1f}  ->  dL = {luminosity_distance(z_test)/1e3:.2f} Gpc")


---
## 5. Population-Level Distributions

Analysing the full GWTC as a **statistical ensemble** reveals the underlying astrophysical population. Key considerations:

1. **Selection effects**: detectors are more sensitive to massive, nearby events. A raw histogram is *not* the true distribution — it reflects a convolution of the true population with the detector response.  
2. **Parameter uncertainty**: each event contributes a *posterior distribution*, not a single point. Population analyses marginalise over these uncertainties.  
3. **Censoring**: marginal events with $p_{\rm astro} < 1$ contribute fractional weights.

In this section we focus on **exploratory visualisation** of the catalog-level properties, deferring rigorous population inference to Section 8.

### 5.1 Mass Spectrum

The black hole mass function has been one of the most exciting outputs of GWTC-3. Key findings:
- A **peak** near $M_\odot \approx 8$–10 (possibly related to core-collapse SN remnants)  
- A possible **feature or bump** around 35 $M_\odot$ (pair-instability mass gap boundary)  
- A **sharp cutoff** above ~45–50 $M_\odot$ in the primary mass distribution (pair-instability gap)  
- Events *above* the gap (e.g. GW190521) may be hierarchical merger products  

### 5.2 Distance and Redshift Distribution

The observed $d_L$ distribution reflects:
- **Cosmic volume** growing as $d_L^2$ for small $z$  
- **Merger rate** evolution with redshift (star formation history)  
- **Detector sensitivity horizon** (cuts off the observed distribution)  

### 5.3 SNR and Detection Sensitivity

The observed SNR distribution peaks near threshold and falls off steeply, consistent with a uniform-in-volume population (rate $\propto \rho^{-4}$ for Euclidean approximation).

### 5.4 Spin Distribution

The observed $\chi_{\rm eff}$ distribution is centred near zero with a slight preference for positive values, consistent with a mixture of aligned and isotropic spin orientations. The width of the distribution constrains the fraction of systems with large spins.


In [ ]:
# ── 5.1  Mass spectrum — detailed plot ────────────────────────────────────
if HAS_PANDAS:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    bins_m = np.linspace(0, 110, 25)
    axes[0].hist(df['m1'], bins=bins_m, alpha=0.65, label=r'$m_1$ (primary)',
                 color='C0', edgecolor='k')
    axes[0].hist(df['m2'], bins=bins_m, alpha=0.55, label=r'$m_2$ (secondary)',
                 color='C1', edgecolor='k')
    axes[0].axvspan(2, 5, alpha=0.12, color='grey', label='Lower mass gap')
    axes[0].axvspan(50, 130, alpha=0.10, color='red', label='Pair-instability gap')
    axes[0].set(xlabel=r'Mass $[M_\odot]$',
                ylabel='Number of events',
                title='Component mass distribution (GWTC-3)',
                xlim=(0, 115))
    axes[0].legend(fontsize=9)

    # Total mass
    bins_mt = np.linspace(0, 200, 25)
    axes[1].hist(df['Mtotal'], bins=bins_mt, color='C2', edgecolor='k', alpha=0.75)
    axes[1].set(xlabel=r'Total mass $M_{\rm tot} = m_1+m_2\,[M_\odot]$',
                ylabel='Number of events',
                title='Total mass distribution (GWTC-3)')

    plt.tight_layout()
    plt.show()


In [ ]:
# ── 5.2  Redshift and distance distribution ────────────────────────────────
if HAS_PANDAS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Distance
    axes[0].hist(df['dL'], bins=20, color='C3', edgecolor='k', alpha=0.8)
    axes[0].set(xlabel=r'Luminosity distance $d_L$ [Mpc]',
                ylabel='Number of events',
                title='Distance distribution')

    # Cumulative N(< z) — simple proxy for redshift evolution
    z_sorted = np.sort(df['z_approx'])
    axes[1].step(z_sorted, np.arange(1, len(z_sorted)+1), where='post',
                 color='C3', lw=2)
    # Euclidean expectation: N(<z) ~ z^3
    z_th = np.linspace(0, 1.4, 200)
    N_Eucl = len(df) * (z_th / z_th.max())**3
    axes[1].plot(z_th, N_Eucl, 'k--', alpha=0.5, label=r'Euclidean $\propto z^3$')
    axes[1].set(xlabel='Approximate redshift $z$',
                ylabel=r'Cumulative $N(<z)$',
                title='Cumulative redshift distribution')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# ── 5.3  SNR distribution: N ~ rho^{-4} expectation ──────────────────────
if HAS_PANDAS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    rho_bins = np.linspace(8, 35, 18)
    axes[0].hist(df['snr'], bins=rho_bins, color='C4', edgecolor='k', alpha=0.8)
    axes[0].axvline(df['snr'].median(), color='r', ls='--',
                    label=f"Median = {df['snr'].median():.1f}")
    axes[0].set(xlabel=r'Network SNR $\rho$',
                ylabel='Number of events',
                title='SNR distribution')
    axes[0].legend()

    # log-log: should be ~ rho^{-4} for uniform-in-volume pop
    rho_vals = np.sort(df['snr'].values)
    rho_log  = np.logspace(np.log10(8), np.log10(35), 100)
    axes[1].loglog(rho_vals, np.arange(len(rho_vals), 0, -1),
                   'o', ms=6, label='Observed (complementary CDF)')
    # Expected rho^{-3} (Euclidean: N(>rho) ~ rho^{-3})
    norm_idx = np.argmin(np.abs(rho_vals - 10))
    rho_ref  = rho_vals[norm_idx]
    N_ref    = len(rho_vals) - norm_idx
    axes[1].loglog(rho_log, N_ref * (rho_ref / rho_log)**3,
                   'k--', alpha=0.6, label=r'$\propto \rho^{-3}$ (Euclidean)')
    axes[1].set(xlabel=r'Network SNR $\rho$',
                ylabel=r'$N(>\rho)$',
                title='Complementary CDF of SNR')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# ── 5.4  Effective spin distribution ──────────────────────────────────────
if HAS_PANDAS:
    fig, ax = plt.subplots(figsize=(9, 5))

    from scipy.stats import gaussian_kde

    kde = gaussian_kde(df['chi_eff'], bw_method=0.3)
    chi_range = np.linspace(-1.0, 1.0, 300)
    ax.fill_between(chi_range, kde(chi_range), alpha=0.3, color='C2')
    ax.plot(chi_range, kde(chi_range), color='C2', lw=2, label='KDE of catalog medians')
    ax.hist(df['chi_eff'], bins=20, density=True, alpha=0.4,
            color='C2', edgecolor='k', label='Histogram')
    ax.axvline(0, color='k', ls='--', alpha=0.6, label=r'$\chi_{\rm eff}=0$')
    ax.axvline(df['chi_eff'].mean(), color='C0', ls='-.',
               label=fr"Mean = {df['chi_eff'].mean():.3f}")
    ax.set(xlabel=r'Effective inspiral spin $\chi_{\rm eff}$',
           ylabel='Probability density',
           title=r'$\chi_{\rm eff}$ distribution (GWTC-3 catalog medians)',
           xlim=(-1, 1))
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"chi_eff statistics:")
    print(f"  mean   = {df['chi_eff'].mean():.4f}")
    print(f"  median = {df['chi_eff'].median():.4f}")
    print(f"  std    = {df['chi_eff'].std():.4f}")
    print(f"  frac > 0: {(df['chi_eff'] > 0).mean():.2%}")


---
## 6. Reproducing "Masses in the Stellar Graveyard"

The **"Masses in the Stellar Graveyard"** figure, produced by LIGO/Virgo/KAGRA, has become one of the most iconic visualisations in modern astrophysics. It places each GW event in a diagram where:

- The **x-axis** shows source mass (in $M_\odot$)
- Each **bar** represents a compact object: $m_1$ (primary) and $m_2$ (secondary)
- Colors distinguish **black holes** (blue/grey) from **neutron stars** (orange) and **ambiguous objects** (teal)
- The **lower mass gap** (2–5 $M_\odot$) and **pair-instability gap** (>~50 $M_\odot$) are shaded
- Events are sorted by total mass

We reproduce an approximate version using our catalog DataFrame.

> **Official figure**: https://media.ligo.northwestern.edu/gallery/mass-plot


In [ ]:
# ── 6. Stellar Graveyard reproduction ─────────────────────────────────────
if HAS_PANDAS:
    # Sort events by total mass
    df_sorted = df.sort_values('Mtotal').reset_index(drop=True)

    def classify(row):
        """Rough source classification based on masses."""
        if row['m2'] < 3.0 and row['m1'] < 3.0:
            return 'BNS'
        elif row['m2'] < 3.0:
            return 'NSBH'
        elif 2.0 < row['m2'] < 5.0 or 2.0 < row['m1'] < 5.0:
            return 'Ambiguous'
        else:
            return 'BBH'

    df_sorted['class'] = df_sorted.apply(classify, axis=1)

    colors = {'BBH': '#4878CF', 'BNS': '#E87722', 'NSBH': '#59A14F', 'Ambiguous': '#B07AA1'}

    fig, ax = plt.subplots(figsize=(14, 10))

    # Shaded regions
    ax.axvspan(2.0, 5.0, alpha=0.08, color='grey', zorder=0)
    ax.axvspan(50.0, 110.0, alpha=0.06, color='red', zorder=0)

    # Plot each event as a horizontal bar connecting m2 to m1
    for idx, row in df_sorted.iterrows():
        col = colors[row['class']]
        y   = idx
        ax.plot([row['m2'], row['m1']], [y, y],
                color=col, lw=4, alpha=0.75, solid_capstyle='round')
        ax.plot(row['m1'], y, 'o', color=col, ms=9, zorder=5)
        ax.plot(row['m2'], y, 's', color=col, ms=7, alpha=0.8, zorder=5)

    # Annotations for famous events
    notable = {
        'GW150914': 'GW150914',
        'GW170817': 'GW170817 (BNS)',
        'GW190521': 'GW190521\n(IMBH!)',
        'GW190814': 'GW190814',
        'GW190412': 'GW190412',
    }
    for ename, label in notable.items():
        if ename in df_sorted['name'].values:
            idx_e = df_sorted.index[df_sorted['name'] == ename][0]
            row_e = df_sorted.loc[idx_e]
            ax.annotate(label,
                        xy=(row_e['m1'], idx_e),
                        xytext=(row_e['m1'] + 3, idx_e),
                        fontsize=7.5,
                        arrowprops=dict(arrowstyle='->', lw=0.8),
                        va='center')

    # Legend
    legend_handles = [mpatches.Patch(color=c, label=k) for k, c in colors.items()]
    legend_handles += [
        mpatches.Patch(color='grey', alpha=0.25, label='Lower mass gap (2–5 M☉)'),
        mpatches.Patch(color='red',  alpha=0.20, label='Pair-instability gap (>50 M☉)'),
    ]
    ax.legend(handles=legend_handles, loc='lower right', fontsize=9)

    ax.set(xlabel=r'Mass $[M_\odot]$',
           ylabel='Events (sorted by total mass)',
           title='Masses in the Stellar Graveyard — GWTC-3\n'
                 r'(circles = $m_1$, squares = $m_2$; bars span [$m_2$, $m_1$])',
           xlim=(0, 112))
    ax.yaxis.set_major_formatter(ticker.NullFormatter())
    ax.tick_params(left=False)

    # Mass scale ticks
    for mass_label, xpos in [(r'$3\,M_\odot$', 3), (r'$50\,M_\odot$', 50)]:
        ax.axvline(xpos, color='k', ls=':', alpha=0.3, lw=0.8)

    plt.tight_layout()
    plt.show()
    print("Source classification summary:")
    print(df_sorted['class'].value_counts().to_string())


---
## 7. Accessing Posterior Samples (HDF5)

### 7.1 HDF5 File Structure

For each GWTC event, LVK releases a **posterior samples HDF5 file** on GWOSC. These contain the full multi-dimensional posterior distribution over the source parameters, as estimated by Bayesian inference (typically using `LALInference`, `Bilby`, or `RIFT`).

A typical file structure (using `h5py`) looks like:

```
GW150914_095045_priordict_pesummary.h5
├── C01:IMRPhenomXP/          # analysis approximant
│   └── posterior_samples     # dataset: shape (N_samples, N_params)
├── C01:SEOBNRv4PHM/
│   └── posterior_samples
└── version                   # catalog version
```

Available parameters typically include:

| Parameter | Description |
|---|---|
| `mass_1_source` | Primary source-frame mass |
| `mass_2_source` | Secondary source-frame mass |
| `chirp_mass` | Detector-frame chirp mass |
| `mass_ratio` | Mass ratio $q = m_2/m_1$ |
| `chi_eff` | Effective spin |
| `chi_p` | Precessing spin |
| `luminosity_distance` | Luminosity distance (Mpc) |
| `redshift` | Source redshift |
| `log_likelihood` | Log-posterior value |
| `ra`, `dec` | Sky location |
| `theta_jn` | Inclination angle |
| `psi` | Polarisation angle |

### 7.2 Loading and Inspecting Posterior Data

GWOSC provides posterior files for all GWTC events. Files can be downloaded via:

```python
import gwosc
urls = gwosc.locate.get_urls('GW150914', catalog='GWTC-3')
```

Or directly from:  
**https://gwosc.org/events/GW150914/**

### 7.3 Corner Plots

A **corner plot** visualises the marginal posteriors for each parameter (diagonal) and their pairwise correlations (off-diagonal). The `corner` package is the standard tool.

### 7.4 Computing Derived Quantities

From the raw posterior samples we can compute any derived quantity, for example:
- **Final mass**: $M_f \approx (m_1 + m_2)(1 - E_{\rm rad})$ where $E_{\rm rad} \approx 0.05$
- **Final spin**: estimated from fits to NR simulations
- **Radiated energy**: $E_{\rm rad} = (m_1 + m_2 - M_f)c^2$
- **Peak luminosity**: from waveform amplitude
- **Remnant classification**: neutron star, BH, or prompt collapse


In [ ]:
# ── 7.1  Working with HDF5 posterior files ────────────────────────────────
#
# In a live environment, download a posterior file from GWOSC and load it:
#
#   import urllib.request
#   url = 'https://gwosc.org/eventapi/json/GWTC-2/GW150914/'
#   # then use gwosc to get the posterior file URL
#
# Here we demonstrate the workflow using a *simulated* posterior to keep
# the notebook self-contained.

print("=" * 60)
print("HDF5 Posterior Samples — Simulated Demonstration")
print("=" * 60)

# Simulate GW150914-like posterior samples
N_samples = 5000

# Correlated multivariate Gaussian for (m1, m2, chi_eff)
mean_vec = np.array([35.6, 30.6, -0.01, 440.0])
# Covariance (rough scale)
cov = np.diag([20.0, 16.0, 0.04, 6000.0])**2
cov[0,1] = cov[1,0] = 0.6 * np.sqrt(cov[0,0] * cov[1,1])  # m1-m2 correlation

samples_raw = rng.multivariate_normal(mean_vec, cov, N_samples)

# Enforce physical constraints: m1 >= m2 > 0
mask = (samples_raw[:,0] >= samples_raw[:,1]) & (samples_raw[:,1] > 3.0)
samples = samples_raw[mask]
print(f"Accepted {len(samples)} / {N_samples} samples after cuts")

m1_s   = samples[:, 0]
m2_s   = samples[:, 1]
chi_s  = np.clip(samples[:, 2], -1, 1)
dL_s   = np.abs(samples[:, 3])
Mc_s   = (m1_s * m2_s)**0.6 / (m1_s + m2_s)**0.2
q_s    = m2_s / m1_s

print("\nPosterior medians and 90% credible intervals:")
for name, arr in [('m1', m1_s), ('m2', m2_s), ('Mc', Mc_s),
                  ('q', q_s), ('chi_eff', chi_s), ('dL', dL_s)]:
    lo, med, hi = np.percentile(arr, [5, 50, 95])
    print(f"  {name:>8s}: {med:7.2f}  [{lo:.2f}, {hi:.2f}]")


In [ ]:
# ── 7.2  Saving simulated posterior to HDF5 ───────────────────────────────
import tempfile, os

if HAS_H5PY:
    tmpfile = os.path.join(tempfile.gettempdir(), 'GW150914_sim_posterior.h5')
    with h5py.File(tmpfile, 'w') as f:
        grp = f.create_group('C01:IMRPhenomXP')
        ps  = grp.create_group('posterior_samples')
        ps.create_dataset('mass_1_source',      data=m1_s)
        ps.create_dataset('mass_2_source',      data=m2_s)
        ps.create_dataset('chirp_mass',         data=Mc_s)
        ps.create_dataset('mass_ratio',         data=q_s)
        ps.create_dataset('chi_eff',            data=chi_s)
        ps.create_dataset('luminosity_distance',data=dL_s)
        f.attrs['event'] = 'GW150914 (simulated)'
        f.attrs['catalog'] = 'GWTC-3'
    print(f"Saved to: {tmpfile}")

    # Re-load and inspect structure
    print("\nHDF5 file structure:")
    with h5py.File(tmpfile, 'r') as f:
        def print_tree(name, obj):
            indent = '  ' * name.count('/')
            if isinstance(obj, h5py.Dataset):
                print(f"{indent}{name}: shape={obj.shape}, dtype={obj.dtype}")
            else:
                print(f"{indent}{name}/")
        f.visititems(print_tree)
else:
    print("h5py not available. Install with: pip install h5py")


In [ ]:
# ── 7.3  Corner plot of posterior samples ─────────────────────────────────
try:
    import corner
    HAS_CORNER = True
except ImportError:
    HAS_CORNER = False
    print("corner not installed — install with: pip install corner")

if HAS_CORNER:
    param_labels = [
        r'$m_1\,[M_\odot]$', r'$m_2\,[M_\odot]$',
        r'$\mathcal{M}\,[M_\odot]$', r'$q$',
        r'$\chi_{\rm eff}$', r'$d_L\,[\rm Mpc]$',
    ]
    data_corner = np.column_stack([m1_s, m2_s, Mc_s, q_s, chi_s, dL_s])
    truths = [35.6, 30.6, 28.3, 0.86, -0.01, 440]

    fig_corner = corner.corner(
        data_corner,
        labels=param_labels,
        truths=truths,
        quantiles=[0.05, 0.5, 0.95],
        show_titles=True,
        title_kwargs={'fontsize': 10},
        truth_color='C1',
        color='C0',
        bins=30,
        smooth=1.0,
    )
    fig_corner.suptitle('GW150914 — Simulated posterior corner plot', y=1.01, fontsize=13)
    plt.show()
else:
    # Fallback: manual 2D marginal plot
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    pairs = [(m1_s, m2_s, r'$m_1$', r'$m_2$'),
             (m1_s, Mc_s, r'$m_1$', r'$\mathcal{M}$'),
             (m1_s, chi_s, r'$m_1$', r'$\chi_{\rm eff}$'),
             (m2_s, q_s, r'$m_2$', r'$q$'),
             (Mc_s, q_s, r'$\mathcal{M}$', r'$q$'),
             (chi_s, dL_s, r'$\chi_{\rm eff}$', r'$d_L$')]
    for ax, (x, y, xl, yl) in zip(axes.ravel(), pairs):
        ax.hexbin(x, y, gridsize=30, cmap='Blues', mincnt=1)
        ax.set(xlabel=xl, ylabel=yl)
    plt.suptitle('GW150914 posterior — 2D marginals (corner not installed)', fontsize=11)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 7.4  Derived quantities from posterior samples ─────────────────────────
# Final mass and radiated energy (NR fit; e.g. Barausse & Rezzolla 2009 approximation)
def final_mass_approx(m1, m2):
    """
    Rough estimate of final BH mass using the Husa et al. (2016) fitting formula.
    Returns Mf in solar masses.
    """
    eta = (m1 * m2) / (m1 + m2)**2  # symmetric mass ratio
    # Radiated energy fraction (Husa et al. 2016, PRD 93 044006, Eq. 22, spin-less limit)
    E_rad_frac = 0.0588 - 0.1274 * eta + 0.2765 * eta**2
    return (m1 + m2) * (1.0 - E_rad_frac)

def final_spin_approx(m1, m2):
    """
    Non-spinning final BH spin (Buonanno et al. 2008, PRD 77 026004, Eq. 8).
    """
    eta = (m1 * m2) / (m1 + m2)**2
    return np.sqrt(12) * eta - 3.871 * eta**2 + 4.028 * eta**3

Mf_s   = final_mass_approx(m1_s, m2_s)
af_s   = final_spin_approx(m1_s, m2_s)
Erad_s = (m1_s + m2_s) - Mf_s   # in solar masses; multiply by Msun*c^2 for joules

# Convert radiated energy to Watts peak (rough)
Msun_kg = 1.989e30
c_m     = 2.998e8
t_merge = 0.01   # approximate merger duration [s]
Erad_J  = Erad_s * Msun_kg * c_m**2

print("Derived quantities from simulated GW150914 posterior:")
print(f"  Mf   = {np.median(Mf_s):.1f}  [{np.percentile(Mf_s,5):.1f}, {np.percentile(Mf_s,95):.1f}] Msun")
print(f"  af   = {np.median(af_s):.3f} [{np.percentile(af_s,5):.3f}, {np.percentile(af_s,95):.3f}]")
print(f"  Erad = {np.median(Erad_s):.2f}  [{np.percentile(Erad_s,5):.2f}, {np.percentile(Erad_s,95):.2f}] Msun")
print(f"  Erad ~ {np.median(Erad_J):.2e} J (for reference: Sun lifetime ~ 1e44 J)")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, arr, label in [
        (axes[0], Mf_s, r'Final mass $M_f\,[M_\odot]$'),
        (axes[1], af_s, r'Final spin $a_f$'),
        (axes[2], Erad_s, r'Radiated energy $E_{\rm rad}\,[M_\odot]$')]:
    ax.hist(arr, bins=30, color='C0', edgecolor='k', alpha=0.75, density=True)
    ax.axvline(np.median(arr), color='C1', ls='--', label='Median')
    ax.set(xlabel=label, ylabel='Density')
    ax.legend(fontsize=9)

plt.suptitle('GW150914 — Derived posterior quantities (simulated)', fontsize=11)
plt.tight_layout()
plt.show()


---
## 8. Advanced Topics

### 8.1 The Mass Gap(s)

Stellar evolution theory predicts the existence of two mass gaps where compact objects should be rare or absent:

**Lower mass gap (2–5 $M_\odot$):**  
Galactic X-ray binary observations found a paucity of compact objects with masses $\sim 2$–5 $M_\odot$ — the region between the maximum NS mass ($\lesssim 2.3\,M_\odot$) and the lightest known stellar-mass BHs ($\gtrsim 5\,M_\odot$). This could reflect:
- Rapid supernova collapse ("failed supernovae")  
- Fallback accretion  
- Selection effects in X-ray binary surveys  

GW190814 ($m_2 \approx 2.6\,M_\odot$) and GW200105 challenge this gap!

**Pair-instability mass gap (~50–130 $M_\odot$):**  
In stars above $\sim 60\,M_\odot$, helium cores above $\sim 30\,M_\odot$ become unstable due to electron-positron pair production. This triggers pulsational or complete pair-instability supernovae (PISN), which *disrupt the star entirely* — leaving no remnant. The resulting gap in the BH mass function extends from $\sim 50$ to $\sim 130\,M_\odot$.

GW190521 ($m_1 \approx 95\,M_\odot$, $m_2 \approx 69\,M_\odot$) has *both* components in the pair-instability gap — possible explanations: hierarchical mergers in dense star clusters, or modified stellar evolution in metal-poor environments.

### 8.2 Population Inference

To infer the true astrophysical population from the observed catalog, we must account for:

1. **Selection effects**: the probability $p(\text{detect} | \theta)$ varies strongly with event parameters (masses and distance).  
2. **Parameter uncertainties**: each event's posterior samples encode measurement uncertainty.  
3. **Hyperparameter inference**: we model the population as $p(\theta | \Lambda)$ parameterised by hyperparameters $\Lambda$ (e.g. power-law index for the mass spectrum).  

The **hierarchical Bayesian** (hyper-posterior) is:

$$
p(\Lambda | \{d_i\}) \propto \pi(\Lambda) \prod_{i=1}^{N} \frac{1}{\xi(\Lambda)} \int p(d_i | \theta_i)\, p(\theta_i | \Lambda)\, d\theta_i
$$

where $\xi(\Lambda) = \int p(\text{detect} | \theta)\, p(\theta | \Lambda)\, d\theta$ is the **selection factor** (normalises for detectability).

**Population models used in LVK papers:**

| Model | Description |
|---|---|
| Power Law + Peak | $m_1 \sim$ truncated power law + Gaussian peak |
| Broken Power Law | Two-segment power law for $m_1$ |
| Multi-peak | Power law + multiple Gaussian peaks |
| Flexible (non-parametric) | Gaussian process or mixture models |

Tools: `gwpopulation` (Python), `popstock`, `GWPopCosmo`.

### 8.3 Gravitational-Wave Transient Source Classes

Beyond the main CBC classes, GWTC also includes searches for:

| Source | Expected signal | Status |
|---|---|---|
| Continuous GW (CW) | Sinusoidal from rotating NS | Not detected in O3 |
| Stochastic background | Broadband noise from unresolved BBH | Upper limits set |
| Burst (unmodelled) | Core collapse SN, cosmic strings | Not detected |
| Post-merger (BNS) | kHz oscillations from remnant NS | Not detected |

### 8.4 Standard Sirens and $H_0$

GW events are **absolute distance indicators** (standard sirens). When a redshift is available (from EM counterpart or statistical host-galaxy methods), the Hubble constant can be measured:

$$
H_0 = \frac{v_H}{d_L} \approx \frac{c\, z}{d_L}
$$

GW170817 + kilonova: $H_0 = 70^{+12}_{-8}$ km/s/Mpc (Abbott et al. 2017).  
GWTC-3 dark sirens (statistical method): $H_0 = 68^{+8}_{-6}$ km/s/Mpc (Abbott et al. 2023).

This measurement is independent of both the cosmic distance ladder and the CMB, making GW sirens a key tool in resolving the **Hubble tension**.


In [ ]:
# ── 8.1  Mass gap visualisation ───────────────────────────────────────────
if HAS_PANDAS:
    fig, ax = plt.subplots(figsize=(11, 6))

    # Plot all masses (m1 and m2) as a 2D histogram
    all_masses = np.concatenate([df['m1'].values, df['m2'].values])
    ax.hist(all_masses, bins=np.linspace(0, 110, 35),
            color='C0', edgecolor='k', alpha=0.8, label='All component masses')

    # Shade mass gaps
    ax.axvspan(2, 5, alpha=0.18, color='orange',
               label='Lower mass gap (2–5 $M_\odot$)')
    ax.axvspan(50, 130, alpha=0.13, color='red',
               label='Pair-instability gap (50–130 $M_\odot$)')

    # Mark neutron star max mass
    ax.axvline(2.3, color='C3', ls='--', lw=1.5,
               label=r'Max NS mass $\approx 2.3\,M_\odot$')

    # Annotate notable objects
    ax.annotate('GW190814\n$m_2\\approx 2.6\\,M_\\odot$',
                xy=(2.6, 2.5), xytext=(6, 4.5),
                arrowprops=dict(arrowstyle='->', lw=0.9), fontsize=9)
    ax.annotate('GW190521\n$m_1\\approx 95\\,M_\\odot$',
                xy=(95, 0.8), xytext=(75, 2.5),
                arrowprops=dict(arrowstyle='->', lw=0.9), fontsize=9)

    ax.set(xlabel=r'Component mass $[M_\odot]$',
           ylabel='Number of components',
           title='Mass distribution & stellar mass gaps (GWTC-3)',
           xlim=(0, 110))
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 8.2  Simple power-law population model ────────────────────────────────
#
# Fit a truncated power law p(m1) ~ m1^{-alpha} to the primary masses
# using a maximum-likelihood estimator (demonstration only — not selection-corrected).

from scipy.optimize import minimize_scalar

if HAS_PANDAS:
    m1_bbh = df.loc[df['class'] == 'BBH', 'm1'].values
    # Apply a mass cut to focus on the main BBH population
    m1_fit = m1_bbh[(m1_bbh > 5) & (m1_bbh < 60)]

    def neg_log_like_powerlaw(alpha, m, mmin=5.0, mmax=60.0):
        """Negative log-likelihood for truncated power law."""
        if alpha == 1.0:
            norm = np.log(mmax / mmin)
        else:
            norm = (mmax**(1-alpha) - mmin**(1-alpha)) / (1 - alpha)
        return -np.sum(-alpha * np.log(m) - np.log(norm))

    result = minimize_scalar(neg_log_like_powerlaw,
                             bounds=(0.5, 5.0), method='bounded',
                             args=(m1_fit,))
    alpha_hat = result.x
    print(f"Best-fit power-law index: alpha = {alpha_hat:.2f}")
    print(f"(LVK O3b population analysis reports alpha ~ 3.5, corrected for selection)")

    # Plot
    fig, ax = plt.subplots(figsize=(9, 5))
    bins_pl = np.linspace(5, 60, 18)
    counts, _, _ = ax.hist(m1_fit, bins=bins_pl, density=True,
                           color='C0', edgecolor='k', alpha=0.75, label='Observed BBH $m_1$')
    m_range = np.linspace(5, 60, 200)
    # Normalised power law
    denom = (60**(1-alpha_hat) - 5**(1-alpha_hat)) / (1-alpha_hat)
    pl_fit = m_range**(-alpha_hat) / denom
    ax.plot(m_range, pl_fit, 'C1-', lw=2.5,
            label=fr'Power law $p\propto m_1^{{-{alpha_hat:.1f}}}$')
    ax.set(xlabel=r'Primary mass $m_1\,[M_\odot]$',
           ylabel='Probability density',
           title='Truncated power-law fit to BBH primary masses')
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 8.4  Standard siren H0 estimator (toy demonstration) ─────────────────
#
# Using the GW170817 measurement as an example of the siren method.

# GW170817 posteriors (from Abbott et al. 2017)
dL_GW170817  = 40.7      # Mpc, median
dL_err       = 8.0       # +/- 1 sigma
z_host_NGC4993 = 0.009680 # redshift of NGC 4993 (from EM)
cz_peculiar  = 310       # km/s (peculiar velocity correction)

# Simulate posterior samples of dL
dL_siren = rng.normal(dL_GW170817, dL_err, 10000)
dL_siren = dL_siren[dL_siren > 0]

# Hubble flow velocity
c_km = 2.998e5
v_H  = c_km * z_host_NGC4993 - cz_peculiar   # km/s

# H0 samples
H0_samples = v_H / dL_siren

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(H0_samples, bins=50, color='C3', edgecolor='k', alpha=0.75, density=True)
ax.axvline(np.median(H0_samples), color='k', ls='--', lw=2,
           label=f"Median H0 = {np.median(H0_samples):.1f} km/s/Mpc")
ax.axvline(67.4, color='C0', ls=':', lw=1.5, label='Planck 2018 (67.4)')
ax.axvline(73.0, color='C1', ls=':', lw=1.5, label='SH0ES (73.0)')
ax.set(xlabel=r'$H_0$ [km/s/Mpc]',
       ylabel='Probability density',
       title=r'Standard siren $H_0$ from GW170817 (toy demonstration)',
       xlim=(40, 120))
lo, hi = np.percentile(H0_samples, [5, 95])
ax.axvspan(lo, hi, alpha=0.15, color='C3', label=f"90% CI: [{lo:.0f}, {hi:.0f}]")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"H0 = {np.median(H0_samples):.1f} +{hi-np.median(H0_samples):.1f} / "
      f"-{np.median(H0_samples)-lo:.1f} km/s/Mpc")


---
## 9. Student Exercises

Work through the following exercises using the skills and code developed in this lesson.

---

### Exercise 1 — Catalog queries with `gwosc`  *(beginner)*

a) Using `gwosc`, list all events in the GWTC-2 catalog and count how many have $p_{\rm astro} > 0.99$.  
b) Retrieve the GPS time and detector list for the five most significant (loudest) events in GWTC-3.  
c) For GW170817, determine which detectors were online and download the corresponding strain data URLs.

---

### Exercise 2 — Mass parameter relations  *(beginner–intermediate)*

a) Write a function `chirp_mass_from_freq(f_dot, f)` that computes the chirp mass from the observed GW frequency $f$ and its time derivative $\dot{f}$, using:
$$
\mathcal{M} = \frac{c^3}{G} \left( \frac{5}{96} \pi^{-8/3} f^{-11/3} \dot{f} \right)^{3/5}
$$
b) Verify that for GW150914 ($f \approx 35$ Hz, $\dot{f} \approx 3.5$ Hz/s near merger), you recover $\mathcal{M} \approx 28\,M_\odot$.  
c) Plot $\mathcal{M}$ as a function of $f$ for $f = 10$–200 Hz, assuming a constant $\dot{f}$ from GW150914.

---

### Exercise 3 — Population statistics  *(intermediate)*

Using the catalog DataFrame constructed in Section 3:

a) Compute and plot the **total mass** $M_{\rm tot} = m_1 + m_2$ distribution for all events.  
b) Separate the BBH, BNS, and NSBH populations and compute the median chirp mass for each.  
c) Is there a correlation between $\chi_{\rm eff}$ and $q$? Make a scatter plot and compute the Pearson correlation coefficient.  
d) Plot the observed merger rate per unit comoving volume as a function of approximate redshift (you may use the $d_L \to z$ conversion from Section 4).

---

### Exercise 4 — Reproducing the stellar graveyard  *(intermediate)*

a) Enhance the stellar graveyard plot from Section 6:  
   - Add 90% credible-interval error bars on $m_1$ and $m_2$ (you may use asymmetric Gaussian widths).  
   - Color each bar by the effective spin $\chi_{\rm eff}$ using a diverging colormap centered on zero.  
   - Add annotations for all NSBH events and both BNS events.  
b) Reproduce the $m_1$–$m_2$ plane density plot (2D histogram or kernel density estimate), marking the equal-mass line $m_1 = m_2$ and the pair-instability gap.

---

### Exercise 5 — Posterior samples analysis  *(intermediate–advanced)*

Download the posterior samples HDF5 file for GW190412 from GWOSC:  
https://gwosc.org/events/GW190412/

a) Load the file with `h5py` and list all available parameter datasets.  
b) Plot the marginal posteriors for $m_1$, $m_2$, $q$, and $\chi_{\rm eff}$.  
c) Compute the posterior predictive distribution of the chirp mass $\mathcal{M}$ from the $m_1$, $m_2$ samples and compare it to the directly sampled `chirp_mass` column.  
d) GW190412 is notable for its mass asymmetry. Plot the 2D posterior in the ($m_1$, $q$) plane and compare with GW150914 (using the simulated posterior from Section 7).

---

### Exercise 6 — Power-law mass spectrum  *(advanced)*

a) Fit a truncated power law $p(m_1) \propto m_1^{-\alpha}$ to the BBH primary masses using maximum likelihood estimation (cf. Section 8.2), for two different mass ranges: [5, 40] and [5, 60] $M_\odot$. How does the inferred $\alpha$ depend on the chosen upper cutoff?  
b) Implement a **Bootstrap** analysis (500 resamples) to estimate the uncertainty on $\hat{\alpha}$.  
c) Compare your result with the LVK Power Law + Peak model: $\alpha \approx 3.5$, $m_{\rm max} \approx 87\,M_\odot$ (Abbott et al. 2023, GWTC-3 populations paper). Discuss sources of systematic bias in your uncorrected fit.

---

### Exercise 7 — Standard sirens  *(advanced)*

a) Using the toy $H_0$ estimator from Section 8.4, explore how the uncertainty on $H_0$ depends on the number of GW standard siren events. Simulate $N = 1, 5, 10, 50$ events at $z \approx 0.01$ with 20% distance uncertainties and plot $\sigma_{H_0}$ vs. $N$.  
b) The O4 run is expected to detect $\sim 1$ BNS per month. Estimate the constraint on $H_0$ achievable with 2 years of O4 data, assuming $\sim 30$% have identified EM counterparts.  
c) Research the **dark siren** method (statistical host galaxy approach) and sketch its key ingredients.

---

### Challenge Exercise — Mini population study

Using `gwpopulation` (install with `pip install gwpopulation`) or pure NumPy/SciPy:

1. Implement a simple hierarchical Bayesian analysis for the BBH chirp mass distribution, assuming a Gaussian population model $\mathcal{M} \sim \mathcal{N}(\mu, \sigma)$.  
2. Use the median chirp masses from the catalog as point estimates (ignoring measurement uncertainty for simplicity).  
3. Sample the hyperparameters $(\mu, \sigma)$ with MCMC (e.g. `emcee`) and plot the posterior.  
4. Compare your inferred $\mu$ with the catalog median chirp mass.

**Bonus**: include a simple selection effect by weighting each event by $1/\rho^3$ (inverse-volume proxy).


---
## 10. References

### Catalog Papers

1. **Abbott et al. (LIGO/Virgo, 2019)** — GWTC-1: A gravitational-wave transient catalog of compact binary mergers observed by LIGO and Virgo.  
   *Phys. Rev. X* 9, 031040. [arXiv:1811.12907](https://arxiv.org/abs/1811.12907)

2. **Abbott et al. (LIGO/Virgo, 2021)** — GWTC-2: Compact binary coalescences observed by LIGO and Virgo during the first half of the third observing run.  
   *Phys. Rev. X* 11, 021053. [arXiv:2010.14527](https://arxiv.org/abs/2010.14527)

3. **Abbott et al. (LIGO/Virgo, 2022)** — GWTC-2.1: Deep extended catalog of compact binary coalescences observed by LIGO and Virgo.  
   *Phys. Rev. D* 109, 022001. [arXiv:2108.01045](https://arxiv.org/abs/2108.01045)

4. **Abbott et al. (LVK, 2023)** — GWTC-3: Compact binary coalescences observed by LIGO and Virgo during the second part of the third observing run.  
   *Phys. Rev. X* 13, 041039. [arXiv:2111.03606](https://arxiv.org/abs/2111.03606)

### Population and Astrophysics

5. **Abbott et al. (LVK, 2023)** — Population of merging compact binaries inferred using gravitational waves through GWTC-3.  
   *Phys. Rev. X* 13, 011048. [arXiv:2111.03634](https://arxiv.org/abs/2111.03634)

6. **Fishbach & Holz (2020)** — Where Are LIGO's Big Black Holes?  
   *ApJL* 891, L27. [arXiv:1905.12669](https://arxiv.org/abs/1905.12669)

7. **Talbot & Thrane (2018)** — Measuring the binary black hole mass spectrum with an astrophysically motivated parameterization.  
   *ApJ* 856, 173. [arXiv:1801.02699](https://arxiv.org/abs/1801.02699)

8. **Farr et al. (2011)** — The Mass Distribution of Stellar-mass Black Holes.  
   *ApJ* 741, 103. [arXiv:1011.1459](https://arxiv.org/abs/1011.1459)

### Software and Data

9. **`gwosc` Python package** — GWOSC API client.  
   https://gwosc.readthedocs.io/

10. **`pycbc`** — Gravitational-wave data analysis toolkit.  
    https://pycbc.org  
    Nitz et al. 2020, [arXiv:2001.09310](https://arxiv.org/abs/2001.09310)

11. **`gwpy`** — Python package for gravitational-wave astrophysics.  
    https://gwpy.github.io

12. **`gwpopulation`** — Hierarchical Bayesian population inference for GW.  
    Talbot et al. 2019. https://github.com/ColmTalbot/gwpopulation

13. **`corner`** — Corner plots for MCMC posterior visualisation.  
    Foreman-Mackey (2016). https://corner.readthedocs.io

14. **GWOSC Data Portal** — Gravitational Wave Open Science Center.  
    https://gwosc.org

15. **GWTC-3 Data Release** — Posterior samples, search results, sky maps.  
    https://gwosc.org/GWTC-3/

### Foundational Theory

16. **Maggiore (2007)** — *Gravitational Waves: Theory and Experiments*, Vol. 1. Oxford University Press.

17. **Sathyaprakash & Schutz (2009)** — Physics, Astrophysics and Cosmology with Gravitational Waves.  
    *Living Rev. Relativity* 12, 2. [arXiv:0903.0338](https://arxiv.org/abs/0903.0338)

18. **Vitale et al. (2022)** — Inferring the properties of a population of compact binaries in presence of selection effects.  
    In: *Handbook of Gravitational Wave Astronomy*. [arXiv:2007.05579](https://arxiv.org/abs/2007.05579)

19. **Abbott et al. (2017)** — A gravitational-wave standard siren measurement of the Hubble constant.  
    *Nature* 551, 85–88. [arXiv:1710.05835](https://arxiv.org/abs/1710.05835)

---

*End of Lesson 5.*  
**Next**: Lesson 6 — Matched Filtering & Signal Detection.
